In [ ]:
# %% [markdown]
# # REHAB24-6: 3D World Landmarks & Calibration Validation (`chamodi-week2`)
# මෙම Google Colab ගොනුව මගින් ඔබගේ **`chamodi-week2`** branch එකෙහි ඇති **3D World Landmarks (`angles_world.py`)** සහ **Calibration (`calibration.py`)** කේත, **REHAB24-6 3D OptiTrack Ground Truth** දත්ත සමඟ ස්වයංක්‍රීයව පරීක්ෂා කරයි.
# - **නැවත Dataset එක Upload කිරීමට අවශ්‍ය නැත:** ඔබගේ Google Drive එකේ දැනටමත් ඇති REHAB24 ගොනු මෙමගින් ස්වයංක්‍රීයව සොයාගනී.
# - **කිසිදු වෙනස්කමක් කිරීමට අවශ්‍ය නැත:** ඉහළ සිට පහළට Cells 1 සිට 6 දක්වා පිළිවෙළින් Run කරන්න.

# %%
# ==============================================================================
# STEP 1: Python 3.12 Virtual Environment for MediaPipe (Colab Compatibility)
# Colab හි MediaPipe 0.10.14 නිවැරදිව ධාවනය කිරීමට Python 3.12 පරිසරයක් සකසයි.
# ==============================================================================
import os, subprocess

MP_PY = "/content/mpenv/bin/python"
if not os.path.exists(MP_PY):
  print("Installing uv and creating Python 3.12 environment for MediaPipe...")
  subprocess.run(["pip", "install", "-q", "uv"], check=True)
  subprocess.run(["uv", "venv", "--python", "3.12", "/content/mpenv"], check=True)
  subprocess.run(
      [
          "uv",
          "pip",
          "install",
          "--python",
          MP_PY,
          "numpy==1.26.4",
          "mediapipe==0.10.14",
          "pyyaml",
          "pandas",
          "opencv-python",
          "matplotlib",
          "scikit-learn",
          "scipy",
      ],
      check=True,
  )
print("MediaPipe Python 3.12 environment is ready at:", MP_PY)

# %%
# ==============================================================================
# STEP 2: Clone 'chamodi-week2' Branch & Mount Google Drive
# ඔබගේ 3D කේත ඇති chamodi-week2 branch එක ලබාගෙන Google Drive සම්බන්ධ කරයි.
# ==============================================================================
import glob, os, re, shutil, subprocess, sys, time, zipfile

REPO_DIR = "/content/repo"
BRANCH = "chamodi-week2"
REPO_URL = "https://github.com/suvini2001/AI-based-Physiotherapy-project-.git"

# දැනටමත් වෙනත් branch එකක් තිබේ නම් එය ඉවත් කර chamodi-week2 clone කිරීම
if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

for p in (".", "capture", "pose", "vision"):
  abs_p = os.path.abspath(p)
  if abs_p not in sys.path:
    sys.path.insert(0, abs_p)

print("Checked out branch:", BRANCH)

# ---------------- Options (සැකසුම්) ----------------
MAX_VIDEOS = None  # None = සියලුම arm වීඩියෝ පරීක්ෂා කරයි (ඉක්මන් පරීක්ෂාවකට 2 ලෙස දිය හැක)
EXERCISES = [1, 2, 3]  # Ex1: Arm Abduction, Ex2: Arm VW, Ex3: Table Push-ups
THRESHOLD = 0.70  # Landmark visibility සීමාව (0.70)
MARGIN = 45  # Time lag සෙවීමට ව්‍යායාමයට පෙර සහ පසු අමතර frames ගණන
MAX_LAG = 30  # උපරිම කාල පරතරය (frames)
MODEL_COMPLEXITY = 1  # MediaPipe model complexity (1 = default)
USE_DRIVE = True  # Google Drive හි cache සහ results තැන්පත් කිරීම

# OptiTrack 3D Skeleton හි අත්වල සන්ධි නාම:
# RightArm = උරහිස (Shoulder), RightForeArm = වැලමිට (Elbow), RightHand = මැණික් කටුව (Wrist)
ARMS_OVERRIDE = {
    "right": ("RightArm", "RightForeArm", "RightHand"),
    "left": ("LeftArm", "LeftForeArm", "LeftHand"),
}
COL = {"video": None, "exercise": None, "start": None, "end": None}

DATA = "/content/rehab24"
CACHE = "/content/cache_3d"
RESULTS = "/content/results_3d"

if USE_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  # 3D දත්ත සඳහා වෙනම ෆෝල්ඩර (පැරණි 2D cache එකට බාධාවක් නොවේ)
  CACHE = "/content/drive/MyDrive/rehab24_cache_3d"
  RESULTS = "/content/drive/MyDrive/rehab24_results_3d"

for d in (DATA, CACHE, RESULTS):
  os.makedirs(d, exist_ok=True)

print("DATA dir   :", DATA)
print("CACHE dir  :", CACHE)
print("RESULTS dir:", RESULTS)

# %%
# ==============================================================================
# STEP 3: Locate Existing Dataset in Google Drive (or Fetch Missing 3D Truth)
# ඔබගේ Google Drive හි දැනටමත් ඇති dataset ගොනු ස්වයංක්‍රීයව සොයා සම්බන්ධ කරයි.
# ==============================================================================
import urllib.request

ZENODO_BASE = "https://zenodo.org/records/13305826/files"


def find_in_drive(filename):
  """Google Drive තුළ දැනටමත් ගොනුවක් තිබේදැයි සොයයි."""
  if not os.path.exists("/content/drive/MyDrive"):
    return None
  matches = glob.glob(f"/content/drive/MyDrive/**/{filename}", recursive=True)
  return matches[0] if matches else None


def ensure_file(filename, url_name=None):
  """DATA ෆෝල්ඩරයේ ගොනුව නැතිනම් Drive එකෙන් link කරයි, නැතහොත් Zenodo වෙතින් බාගත කරයි."""
  target = os.path.join(DATA, filename)
  if os.path.exists(target) and os.path.getsize(target) > 0:
    print(f"[OK] Found in DATA: {filename}")
    return target

  drive_hit = find_in_drive(filename)
  if drive_hit:
    print(f"[Drive] Found {filename} at {drive_hit} -> linking to {target}")
    if not os.path.exists(target):
      os.symlink(drive_hit, target)
    return target

  dl_name = url_name or filename
  url = f"{ZENODO_BASE}/{dl_name}?download=1"
  print(f"[Download] Fetching {dl_name} from Zenodo...")
  urllib.request.urlretrieve(url, target)
  return target


# 1. Segmentation.csv සහ joints_names.txt පරීක්ෂා කිරීම
seg_path = find_in_drive("Segmentation.csv") or find_in_drive("Segmentation.txt")
if seg_path and not os.path.exists(
    os.path.join(DATA, os.path.basename(seg_path))
):
  shutil.copy(seg_path, os.path.join(DATA, os.path.basename(seg_path)))
elif not glob.glob(f"{DATA}/Segmentation.*"):
  try:
    ensure_file("Segmentation.csv")
  except Exception:
    ensure_file("Segmentation.txt")

jn_path = find_in_drive("joints_names.txt") or find_in_drive("joint_names.txt")
if jn_path and not os.path.exists(os.path.join(DATA, os.path.basename(jn_path))):
  shutil.copy(jn_path, os.path.join(DATA, os.path.basename(jn_path)))
elif not glob.glob(f"{DATA}/joint*_names.txt"):
  ensure_file("joints_names.txt")

# 2. Videos ෆෝල්ඩරය පරීක්ෂා කිරීම
videos_dir = os.path.join(DATA, "videos")
if not os.path.exists(videos_dir) or not glob.glob(
    f"{videos_dir}/**/*.*", recursive=True
):
  drive_mp4s = glob.glob(
      "/content/drive/MyDrive/**/PM_*-cam*.mp4", recursive=True
  )
  if drive_mp4s:
    drive_vid_folder = os.path.dirname(drive_mp4s[0])
    print(f"[Drive] Linking existing videos folder: {drive_vid_folder}")
    if os.path.exists(videos_dir):
      shutil.rmtree(videos_dir)
    os.symlink(drive_vid_folder, videos_dir)
  else:
    z_vid = ensure_file("videos.zip")
    print("Extracting videos.zip...")
    with zipfile.ZipFile(z_vid, "r") as zf:
      zf.extractall(DATA)

# 3. 3D Ground Truth (3d_joints) ෆෝල්ඩරය පරීක්ෂා කිරීම (3D Evaluation සඳහා අත්‍යවශ්‍ය වේ)
joints3d_dir = os.path.join(DATA, "3d_joints")
if not os.path.exists(joints3d_dir) or not glob.glob(
    f"{joints3d_dir}/**/*.*", recursive=True
):
  drive_3d = glob.glob("/content/drive/MyDrive/**/3d_joints", recursive=True)
  if drive_3d and os.path.isdir(drive_3d[0]) and os.listdir(drive_3d[0]):
    print(f"[Drive] Linking existing 3d_joints folder: {drive_3d[0]}")
    if os.path.exists(joints3d_dir):
      shutil.rmtree(joints3d_dir)
    os.symlink(drive_3d[0], joints3d_dir)
  else:
    z_3d = ensure_file("3d_joints.zip")
    print("Extracting 3d_joints.zip...")
    with zipfile.ZipFile(z_3d, "r") as zf:
      zf.extractall(DATA)

print("Dataset check complete! Contents of DATA:", os.listdir(DATA))

# %%
# ==============================================================================
# STEP 4: Parse Segmentation Metadata & Extract 3D World Landmarks
# Segmentation.csv කියවා සෑම වීඩියෝවකම 3D World Landmarks (.npz) ලබාගනී.
# ==============================================================================
import numpy as np
import pandas as pd

# 1. Joint names කියවීම
jn_files = glob.glob(f"{DATA}/**/*joint*names*.txt", recursive=True)
assert jn_files, "joints_names.txt ගොනුව හමු නොවීය!"
with open(jn_files[0]) as f:
  JOINT_NAMES = [line.strip() for line in f if line.strip()]
print(
    f"Loaded {len(JOINT_NAMES)} OptiTrack 3D joint names:",
    JOINT_NAMES[:8],
    "...",
)

# 2. Segmentation.csv කියවීම
seg_files = glob.glob(f"{DATA}/**/Segmentation.*", recursive=True)
assert seg_files, "Segmentation.csv ගොනුව හමු නොවීය!"
sep = ";" if ";" in open(seg_files[0]).readline() else ","
seg_df = pd.read_csv(seg_files[0], sep=sep)
print("Segmentation columns:", list(seg_df.columns))

cols_lower = {c.lower(): c for c in seg_df.columns}


def pick_col(key, candidates):
  if COL.get(key):
    return COL[key]
  for cand in candidates:
    for cl, orig in cols_lower.items():
      if cand in cl:
        return orig
  raise KeyError(
      f"Could not guess column for {key} from {list(seg_df.columns)}"
  )


c_vid = pick_col("video", ["video", "file", "rec", "name", "id"])
c_ex = pick_col("exercise", ["ex", "activity", "type", "class"])
c_start = pick_col("start", ["start", "begin", "first", "f0", "from"])
c_end = pick_col("end", ["end", "stop", "last", "f1", "to"])

c_view = None
for cl, orig in cols_lower.items():
  if any(k in cl for k in ["view", "angle", "cam_pos", "orientation"]):
    c_view = orig
    break

all_videos = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f"{DATA}/videos/**/*.*", recursive=True)
    if p.lower().endswith((".mp4", ".avi", ".mov"))
}
all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f"{DATA}/3d_joints/**/*.*", recursive=True)
    if p.lower().endswith((".npy", ".txt", ".csv"))
}

print(
    f"Indexed {len(all_videos)} video files and {len(all_3d)} 3D ground-truth"
    " files."
)

seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []
for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
  vid_str = str(vid_id).strip()
  base_id = vid_str.split("-")[0]
  f0 = max(0, int(grp[c_start].min()) - MARGIN)
  f1 = int(grp[c_end].max()) + MARGIN
  view_val = str(grp[c_view].iloc[0]) if c_view else "unknown"

  matched_vids = [
      k
      for k in all_videos
      if k == vid_str
      or k.startswith(vid_str + "-")
      or k.startswith(base_id + "-")
  ]
  truth_key = next((k for k in [vid_str, base_id] if k in all_3d), None)
  if not truth_key:
    truth_key = next((k for k in all_3d if k.startswith(base_id)), None)

  if not truth_key:
    continue

  for mv in sorted(set(matched_vids)):
    recordings.append({
        "rec_id": f"{mv}_ex{int(ex_id)}",
        "video_key": mv,
        "video_path": all_videos[mv],
        "truth_path": all_3d[truth_key],
        "exercise": int(ex_id),
        "f0": f0,
        "f1": f1,
        "view_raw": view_val,
    })

uniq_recs = {r["rec_id"]: r for r in recordings}
recordings = list(uniq_recs.values())
if MAX_VIDEOS is not None:
  recordings = recordings[:MAX_VIDEOS]

print(f"Prepared {len(recordings)} arm exercise recordings for 3D evaluation.")


def get_3d_landmarks(rec):
  """extract_world.py භාවිතා කර 2D (lm) සහ 3D (world) landmarks ලබාගනී.

  දැනටමත් Drive cache එකේ තිබේ නම් කෙලින්ම load කරයි.
  """
  cache_path = os.path.join(
      CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"
  )
  need_extract = True
  if os.path.exists(cache_path):
    try:
      d = np.load(cache_path)
      if "world" in d and "lm" in d:
        need_extract = False
    except Exception:
      need_extract = True

  if need_extract:
    cmd = [
        MP_PY,
        "extract_world.py",
        rec["video_path"],
        str(rec["f0"]),
        str(rec["f1"]),
        cache_path,
        str(MODEL_COMPLEXITY),
    ]
    subprocess.run(cmd, check=True)

  d = np.load(cache_path)
  return d["lm"], d["world"], tuple(d["size"]), float(d["fps"])


# %%
# ==============================================================================
# STEP 5: Run 3D Benchmark Evaluation (Variant B: 3D World & Variant D: Calibrated)
# ඔබගේ angles_world.py, calibration.py සහ benchmark_common.py මගින් 3D කෝණ පරීක්ෂා කරයි.
# ==============================================================================
import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from calibration import CalibrationModel
from filters import smooth_butterworth
from gap_fill import fill_gaps
from visibility import mask_low_visibility


def infer_view(rec):
  """කැමරා කෝණය (Front, Half-profile, Profile, Side) හඳුනාගැනීම."""
  v = str(rec.get("view_raw", "")).lower()
  if "half" in v:
    return "Half-profile"
  if "front" in v or "0" == v:
    return "Front"
  if "prof" in v:
    return "Profile"
  if "side" in v or "90" in v:
    return "Side"
  if "cam17" in rec["video_key"].lower():
    return "Front"
  if "cam18" in rec["video_key"].lower():
    return "Side"
  return "Front"


rows_A, rows_B = [], []
aligned_cache = []  # Leave-one-person-out Calibration (Variant D) සඳහා

for idx, rec in enumerate(recordings, 1):
  try:
    t0 = time.time()
    lm, world, size, fps = get_3d_landmarks(rec)
    truth_arr = bc.load_truth_array(rec["truth_path"], JOINT_NAMES)
    truth_win = truth_arr[rec["f0"] : rec["f0"] + len(world)]

    # 1. Visibility අඩු (0.70 ට අඩු) frames ඉවත් කිරීම
    lm_masked = mask_low_visibility(lm, threshold=THRESHOLD)
    world_masked = mask_world_like(world, lm_masked)

    for side in ("right", "left"):
      arm_joints = ARMS_OVERRIDE[side]
      # OptiTrack 3D Ground Truth වැලමිට කෝණය
      gt_3d = bc.truth_elbow_3d(truth_win, JOINT_NAMES, arm_joints, use_dims=3)

      # --- Variant B: 3D World Landmarks Angle (angles_world.py) ---
      pred_3d_raw = elbow_angle_3d(world_masked, side=side)
      pred_3d_fill = fill_gaps(pred_3d_raw, max_gap=15)
      pred_3d = (
          smooth_butterworth(pred_3d_fill, fps=fps, cutoff=6.0)
          if np.isfinite(pred_3d_fill).sum() > 15
          else pred_3d_fill
      )

      # --- Variant A: 2D Baseline Angle (angles.py) සංසන්දනය සඳහා ---
      if hasattr(angles_2d, "elbow_angle"):
        pred_2d_raw = angles_2d.elbow_angle(lm_masked, side=side, size=size)
      else:
        pred_2d_raw = pred_3d_raw
      pred_2d_fill = fill_gaps(pred_2d_raw, max_gap=15)
      pred_2d = (
          smooth_butterworth(pred_2d_fill, fps=fps, cutoff=6.0)
          if np.isfinite(pred_2d_fill).sum() > 15
          else pred_2d_fill
      )

      meta = {
          "recording": f"{rec['rec_id']}_{side}",
          "video": rec["video_key"],
          "person": bc.person_of(rec["video_key"]),
          "exercise": rec["exercise"],
          "side": side,
          "view": infer_view(rec),
          "max_lag": MAX_LAG,
      }

      row_b = bc.make_row("B_3D_World", pred_3d, gt_3d, meta)
      row_a = bc.make_row("A_2D_Base", pred_2d, gt_3d, meta)
      rows_B.append(row_b)
      rows_A.append(row_a)

      lag = bc.best_lag(pred_3d, gt_3d, max_lag=MAX_LAG)
      p_al, t_al = bc.overlap(pred_3d, gt_3d, lag)
      aligned_cache.append({"meta": meta, "pred_3d": p_al, "truth_3d": t_al})

    print(
        f"[{idx}/{len(recordings)}] {rec['rec_id']} done in"
        f" {time.time()-t0:.1f}s | 3D MAE (R/L): {rows_B[-2]['mae']:.1f}°,"
        f" {rows_B[-1]['mae']:.1f}°"
    )
  except Exception as e:
    print(f"[{idx}/{len(recordings)}] Skipped {rec['rec_id']} due to: {e}")

# Leave-One-Person-Out Calibration (Variant D: 3D + CalibrationModel)
rows_D = []
for item in aligned_cache:
  test_person = item["meta"]["person"]
  train_p = [
      x["pred_3d"]
      for x in aligned_cache
      if x["meta"]["person"] != test_person
  ]
  train_t = [
      x["truth_3d"]
      for x in aligned_cache
      if x["meta"]["person"] != test_person
  ]
  if train_p:
    p_cat = np.concatenate(train_p)
    t_cat = np.concatenate(train_t)
    valid = np.isfinite(p_cat) & np.isfinite(t_cat)
    if valid.sum() > 20:
      cal = CalibrationModel().fit(p_cat[valid], t_cat[valid])
      pred_cal = cal.predict(item["pred_3d"])
    else:
      pred_cal = item["pred_3d"]
  else:
    pred_cal = item["pred_3d"]

  meta_d = dict(item["meta"], max_lag=0)
  rows_D.append(
      bc.make_row("D_3D_Calibrated", pred_cal, item["truth_3d"], meta_d)
  )

bc.save_rows(rows_A, os.path.join(RESULTS, "variant_A_2d.csv"))
bc.save_rows(rows_B, os.path.join(RESULTS, "variant_B_3d_world.csv"))
bc.save_rows(rows_D, os.path.join(RESULTS, "variant_D_3d_calibrated.csv"))
print("Saved all 3D benchmark CSV files to:", RESULTS)

# %%
# ==============================================================================
# STEP 6: Summary Tables & Markdown Report for notes.md
# කැමරා කෝණය අනුව 2D (Variant A), 3D (Variant B) සහ Calibrated 3D (Variant D) සංසන්දනය.
# ==============================================================================
import benchmark_common as bc
import matplotlib.pyplot as plt
import pandas as pd

df_all = bc.load_variants(os.path.join(RESULTS, "variant_*.csv"))
print("=== Summary Metrics by Variant and Camera View (Coverage >= 50%) ===")
sum_df = bc.summary(df_all, min_coverage=50.0)
print(sum_df)

print("\n=== Markdown Table (Copy and paste directly into notes.md) ===")
md_table = bc.to_markdown(bc.results_table(df_all, min_coverage=50.0))
print(md_table)

sum_df.to_csv(os.path.join(RESULTS, "summary_3d_by_view.csv"), index=False)

if not sum_df.empty and "view" in sum_df.columns and "mae" in sum_df.columns:
  fig, ax = plt.subplots(figsize=(8, 4.5))
  piv = sum_df.pivot(index="view", columns="variant", values="mae")
  piv.plot(kind="bar", ax=ax, rot=0)
  ax.set_ylabel("Median MAE (degrees)")
  ax.set_title("REHAB24-6 Elbow Angle Error: 2D vs 3D World vs 3D Calibrated")
  ax.grid(axis="y", alpha=0.3)
  plt.tight_layout()
  plot_path = os.path.join(RESULTS, "mae_3d_comparison.png")
  plt.savefig(plot_path, dpi=150)
  plt.show()
  print("Saved comparison chart to:", plot_path)